# Push-T × JEPA + **SIGReg officiel** (LeJEPA) — face à DINO-WM (SR 0.90)

World model **JEPA conjoint** (encodeur ViT + dynamique conditionnée par l'action), anti-collapse = **SIGReg du package `lejepa`** (aucun slot, aucune reconstruction, aucun décodeur). Planification **purement latente** (protocole DINO-WM).

Workflow : **1** (code+deps) → **2** (Drive) → **3** collecte (une fois) → **4** entraînement → **5** reprise → **6** éval MPC/oracle/aléa (coût latent) → **7** éval coût latent-sans-agent.

In [ ]:
# 1) Code + dépendances (gym-pusht + lejepa officiel)
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install --quiet gym-pusht 'pymunk<7'
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
import torch, lejepa
t = lejepa.univariate.EppsPulley(t_max=3, n_points=17)
f = lejepa.multivariate.SlicingUnivariateTest(univariate_test=t, num_slices=1024)
z = torch.randn(128,128, requires_grad=True); f(z).backward()
print('OK — lejepa grad ok =', z.grad is not None)

In [ ]:
# 2) Drive (données + checkpoints persistants)
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('pas de Drive:', e)

In [ ]:
# 3) COLLECTE (une fois — sauter si pusht_data.npz est déjà sur Drive)
#!python -u pusht_data.py --n 10000 --T 6

In [ ]:
# 4) ENTRAÎNEMENT depuis zéro (JEPA conjoint + SIGReg rw=1.0)
!python -u pusht_jepa.py --steps 30000 --bs 32 --rw 1.0

In [ ]:
# 5) REPRISE du checkpoint (architecture auto-lue)
#!python -u pusht_jepa.py --resume 1 --steps 20000

In [ ]:
# 6) ÉVAL — PLANIF latente pleine (fidèle DINO-WM) : MPC vs ORACLE vs aléatoire
#    but par image, ≤100 pas, succès = coverage officiel (>0.95)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent

In [ ]:
# 7) ÉVAL — variante coût 'sans agent' (tokens bleus du but exclus)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent_noagent